# Lektion 03 - Agentur-Designmuster

In dieser Lektion untersuchen wir drei grundlegende Designmuster für den Aufbau effektiver KI-Agenten:

1. **Klare Agentenanweisungen** — Verfassen präziser, rollen-definierender Aufforderungen, die das Verhalten des Agenten steuern
2. **Strukturierte Ausgabe mit Pydantic-Modellen** — Sicherstellen, dass Agenten vorhersehbare, validierte Daten zurückgeben
3. **Single-Responsibility-Agenten** — Entwerfen fokussierter Agenten, die jeweils eine Sache gut erledigen

Wir wenden jedes Muster auf ein **Reiseziel-Empfehlungsszenario** an und bauen schrittweise ein System auf, das Ziele vorschlagen, Verfügbarkeiten prüfen und Logistik handhaben kann.


## Einrichtung


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Muster 1: Klare Agenten-Anweisungen

Das wirkungsvollste Muster ist auch das einfachste: klare, detaillierte Anweisungen für Ihren Agenten zu schreiben.

Gute Anweisungen definieren:
- **Wer** der Agent ist (Persona und Ton)
- **Was** er tun soll (Schritt-für-Schritt-Verantwortlichkeiten)
- **Wie** er sich verhalten soll (Beschränkungen und Stil)

Unten erstellen wir einen Reise-Concierge-Agenten mit expliziten Anweisungen, die jede Antwort, die er erzeugt, formen.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Muster 2: Strukturierte Ausgabe mit Pydantic-Modellen

Fließtext ist für Gespräche nützlich, aber nachgelagerte Systeme benötigen strukturierte Daten.
Durch die Kombination von **Pydantic-Modellen** mit einer **Tool-Funktion** können wir:

- Ein genaues Schema für die Ausgabe des Agenten definieren
- Antworten automatisch validieren
- Ergebnisse des Agenten zuverlässig in die Anwendungslogik integrieren

Der Schlüssel zur Durchsetzung ist das Übergeben von `response_format`, wenn wir den Agenten ausführen. Dies zwingt das
Modell dazu, ein validiertes `TravelRecommendations`-Objekt zurückzugeben (verfügbar unter `response.value`)
anstelle von Freitext. Das Tool `get_destination_details` gibt ebenfalls ein typisiertes
`DestinationRecommendation` zurück, sodass die Daten durchgehend strukturiert bleiben.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(
    destination: Annotated[str, "The destination to look up"]
) -> DestinationRecommendation:
    """Get structured details about a vacation destination."""
    details = {
        "Barcelona": DestinationRecommendation(
            destination="Barcelona",
            available=True,
            best_season="May-Jun",
            highlights=["Beach", "Architecture", "Nightlife"],
            estimated_budget_usd=2000,
        ),
        "Tokyo": DestinationRecommendation(
            destination="Tokyo",
            available=True,
            best_season="Mar-Apr",
            highlights=["Culture", "Food", "Technology"],
            estimated_budget_usd=2500,
        ),
        "Cape Town": DestinationRecommendation(
            destination="Cape Town",
            available=False,
            best_season="Nov-Mar",
            highlights=["Nature", "Wine", "Adventure"],
            estimated_budget_usd=1800,
        ),
    }
    return details.get(
        destination,
        DestinationRecommendation(
            destination=destination,
            available=False,
            best_season="Unknown",
            highlights=[],
            estimated_budget_usd=0,
        ),
    )


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

# Passing `response_format` forces the agent to return a validated
# TravelRecommendations object instead of free-form text.
response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget",
    options={"response_format": TravelRecommendations},
)

if response and response.value:
    result: TravelRecommendations = response.value
    for rec in result.recommendations:
        status = "Available" if rec.available else "Not available"
        print(f"{rec.destination} ({status})")
        print(f"  Best season: {rec.best_season}")
        print(f"  Highlights: {', '.join(rec.highlights)}")
        print(f"  Estimated budget: ${rec.estimated_budget_usd}")
        print()
    print(f"Note: {result.personalized_note}")
else:
    print("No validated structured response was returned.")
    print(response)


## Muster 3: Agenten mit einzelner Verantwortung

Komplexe Aufgaben profitieren davon, die Arbeit auf mehrere fokussierte Agenten aufzuteilen, die jeweils eine einzige Verantwortung haben:

- Ein **Zielort-Experte**, der sich mit Orten und Verfügbarkeiten auskennt
- Ein **Logistikplaner**, der Flüge, Hotels und Reisepläne verwaltet

Dies spiegelt das Prinzip der *Trennung der Anliegen* aus dem Software-Engineering wider — jeder Agent ist einfacher zu testen, zu warten und unabhängig zu verbessern.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Zusammenfassung

In dieser Lektion haben wir drei agentenbasierte Entwurfsmuster auf ein Reiseempfehlungsszenario angewendet:

| Muster | Schlüsselidee | Vorteil |
|---|---|---|
| **Klare Anweisungen** | Definieren Sie Persona, Verantwortlichkeiten und Einschränkungen im Voraus | Konsistentes, markenkonformes Agentenverhalten |
| **Strukturierte Ausgabe** | Verwenden Sie Pydantic-Modelle als Antwortformat | Validierte, maschinenlesbare Ergebnisse |
| **Single Responsibility** | Geben Sie jedem Agenten eine fokussierte Aufgabe | Einfacher zu testen, zu warten und zu kombinieren |

Diese Muster lassen sich natürlich kombinieren — Sie können klare Anweisungen mit strukturierter Ausgabe in einem Single-Responsibility-Agent verbinden, um robuste, produktionsreife Systeme zu erstellen.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Haftungsausschluss**:
Dieses Dokument wurde mit dem KI-Übersetzungsdienst [Co-op Translator](https://github.com/Azure/co-op-translator) übersetzt. Obwohl wir uns um Genauigkeit bemühen, beachten Sie bitte, dass automatisierte Übersetzungen Fehler oder Ungenauigkeiten enthalten können. Das Originaldokument in seiner Ursprungssprache gilt als maßgebliche Quelle. Bei kritischen Informationen wird eine professionelle menschliche Übersetzung empfohlen. Wir übernehmen keine Haftung für Missverständnisse oder Fehlinterpretationen, die aus der Verwendung dieser Übersetzung entstehen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
